# 01 · Preprocessing and features

**Purpose.** Show one image through the enhancement pipeline and every descriptor, then build the 60,000 × 426 feature cache.

**Inputs.** `data/raw/` (from notebook 00).

**Outputs.** `data/features/features_all.npy` (float32, 60,000 × 426).

**Syllabus topics.** Colour spaces (YCrCb, HSV, grayscale); histogram processing (CLAHE, colour histograms); convolution and filtering (Gaussian, Gabor); Fourier transform; edge detection (Canny, LoG, DoG); HOG; uniform LBP; Gabor filter banks; Haar DWT.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import os, time

import matplotlib.pyplot as plt
import numpy as np

from mia import visual
from mia.config import FEATURES_PATH, RAW_DIR
from mia.data import CLASS_NAMES, load_pool
from mia.features import BLOCKS, N_DIMS, build_feature_cache
from mia.preprocess import enhance, to_gray

images, labels, _ = load_pool(RAW_DIR)
idx = 7
img = images[idx]
enh, gray = enhance(img), to_gray(enhance(img))
print(f"image {idx}: {CLASS_NAMES[labels[idx]]}")

## Enhancement: CLAHE on the luminance (Y) channel, then a 3 × 3 Gaussian (σ = 0.5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, im, title in zip(axes, [img[..., ::-1], enh[..., ::-1], gray], ["Original (BGR→RGB)", "Enhanced", "Grayscale"]):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Descriptors and frequency / edge views

In [ ]:
views = [("HOG glyphs", visual.hog_glyphs(gray)), ("Uniform LBP codes", visual.lbp_code_map(gray)),
         ("Gabor bank (rows λ=4, 8; cols θ=0..135°)", visual.gabor_grid(gray)), ("Haar DWT, 2 levels", visual.dwt_subbands(gray)),
         ("log |FFT| (centred)", visual.fft_spectrum(gray)), ("Canny (50, 150)", visual.canny(gray)),
         ("LoG (σ = 1)", visual.log_edges(gray)), ("DoG (σ 1 − σ 2)", visual.dog(gray))]
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax, (title, im) in zip(axes.ravel(), views):
    ax.imshow(im, cmap="gray", interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 2.6))
for ax, (name, h) in zip(axes, visual.hsv_hist_data(enh).items()):
    ax.bar(range(len(h)), h, color="#1B3A6B")
    ax.set_title(f"{name} histogram")
    ax.set_xlabel("bin")
axes[0].set_ylabel("fraction of pixels")
plt.tight_layout()
plt.show()

## Feature cache for all 60,000 images
Uses all CPU cores but one; reused on later runs when the shape matches.

In [ ]:
t0 = time.perf_counter()
feats = build_feature_cache(images, FEATURES_PATH, n_jobs=max(1, (os.cpu_count() or 2) - 1))
print(f"feature cache {feats.shape} {feats.dtype} ready in {time.perf_counter() - t0:.1f} s -> {FEATURES_PATH}")

In [ ]:
start = 0
for name, n in BLOCKS:
    print(f"{name:>6}: dims {start:>3}-{start + n - 1:<3} ({n})")
    start += n
assert start == N_DIMS == feats.shape[1] == 426
assert np.all(np.isfinite(feats)), "non-finite feature rows found"
print("all", len(feats), "rows finite")

## Results
The printed block table confirms the 426-dim layout (HOG 324, LBP 40, Gabor 16, DWT 14, HSV 32) and that every one of the 60,000 rows is finite. The cache is the input for every classical model.